# Transaction Data Intelligence — Colab demo

Runs the same pipeline as the Streamlit app (`app.py`), using the project's own `src/` modules directly —
nothing here is reimplemented or duplicated. A GPU runtime speeds up the built-in Transformer and is close to
necessary for the optional Hugging Face / Nemotron backends; everything up through Processing is CPU-friendly.

```
load data -> profile -> E0 -> E1 -> E2 -> train the built-in Transformer -> evaluate -> compare
```

**Before running**: if this notebook was opened directly in Colab (not from a cloned repo), run the setup
cell below first. If you're running it from a local clone or an already-cloned Colab runtime, skip straight to
the second cell.

In [ ]:
# Setup for a fresh Colab runtime: clone the repo and install dependencies.
# Skip this cell if you already have the project checked out in this runtime.
import os

REPO_URL = "<your-repository-url>"  # replace with this project's actual git URL
REPO_DIR = "transaction-data-intelligence"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt
# Optional: only needed for the Hugging Face / Nemotron backends later in this notebook.
# !pip install -q transformers peft accelerate

## 1. Load data

Uses the built-in synthetic demo generator by default (clearly not real data — useful for trying the whole
pipeline without a real dataset). To use a real file instead, upload it in Colab's file browser and set `PATH`
to it, or point `PATH` at any CSV/JSON/JSON Lines file already in the runtime.

In [ ]:
from pathlib import Path

from src.ingestion.demo_data import make_transactions
from src.ingestion.loader import load_dataset
from src.ingestion.roles import detect_roles, schema_for_profiling, target_summary
from src.ingestion.schema_detector import detect_schema
from src.utils.config import ROOT, load_config

PATH = None  # set to a real file path to use your own data instead of the synthetic demo generator

cfg = load_config()
if PATH:
    ds = load_dataset(Path(PATH), metadata_dir=ROOT / cfg["project"]["data_dir"] / "raw" / "_metadata")
    df, dataset_id = ds.df, ds.metadata.dataset_id
else:
    df = make_transactions(n_cards=150, days=120, seed=0)  # same generator the app's "Load synthetic demo data" button uses
    dataset_id = "colab_demo_SYNTHETIC"

schema = detect_schema(df, dataset_id)
roles = detect_roles(df, schema)  # override with detect_roles(df, schema, target="your_target_column") if needed
print(f"{len(df):,} rows, {len(df.columns)} columns")
print(f"target={roles.target!r} task={roles.task!r} entity={roles.entity!r} datetime={roles.datetime!r}")

## 2. Profile

Row/missing/duplicate counts, numeric and categorical summaries, target distribution, early leakage indicators —
nothing here modifies the data.

In [ ]:
from src.profiling.profiler import profile_dataset

ps = schema_for_profiling(schema, roles, df)
profile = profile_dataset(df, ps, top_k=cfg["profiling"]["top_k"], rare_threshold=cfg["profiling"]["rare_threshold"],
                          skew_warning=cfg["profiling"]["skew_warning"])
print(f"{profile.overview['columns']} columns, {len(profile.warnings)} profiling warnings")
if roles.target:
    print(target_summary(df, roles.target, roles.task))

## 3. E0 / E1 / E2

One `DataPreparer` builds all three levels from the *same* split and sample, so any later difference in model
performance comes from the data preparation, not from different rows. `rows="full"` uses every available row
after the split; use a smaller number (e.g. `20000`) for a faster CPU run.

In [ ]:
from src.preprocessing.levels import DataPreparer, infer_roles

level_roles = infer_roles(df, ps, roles)
preparer = DataPreparer(df, ps, level_roles, cfg, rows="full", seed=cfg["project"]["seed"])
split_info = preparer.prepare_split()
print(f"split: {split_info['boundaries']['method']}")
print(split_info["sample"])

levels = {lv: preparer.build(lv) for lv in ["E0", "E1", "E2"]}
for lv, prepared in levels.items():
    print(f"{lv}: {prepared.info['features']} features, {len(prepared.frames['train']):,} train rows")
    for step in prepared.info["steps"]:
        print("   -", step)

## 4. Train the built-in Transformer (one level)

A small, CPU-friendly Transformer encoder — purpose-built to validate that a prepared level produces valid,
learnable model-ready data end to end, not to compete with large pretrained models. This cell trains on E2;
change `LEVEL` to try E0 or E1.

In [ ]:
from src.models.sanity_transformer import SanityTransformerAdapter
from src.utils.hardware import detect_hardware, recommendations

hw = detect_hardware()
for level, msg in recommendations(hw):
    print(f"[{level}] {msg}")

LEVEL = "E2"
adapter = SanityTransformerAdapter(cfg, device=hw["device"])
summary = adapter.train(levels[LEVEL], {})  # {} = use config.yaml's models.sanity_transformer settings
print(f"trained {summary['epochs_run']} epoch(s) in {summary['train_seconds']} s on {summary['device']}")
summary["history"][-1]

## 5. Evaluate

Threshold chosen on validation (best weighted F1), applied unchanged to test. Metrics are computed at the real
class balance as well as unweighted; accuracy is intentionally not the headline metric at this class imbalance.

In [ ]:
checks = adapter.sanity_checks(levels[LEVEL])
assert all(c["status"] != "fail" for c in checks), checks
print(f"{sum(c['status'] == 'pass' for c in checks)}/{len(checks)} sanity checks passed")

results, _, _ = adapter.evaluate(levels[LEVEL])
print(f"threshold (from validation): {results['threshold_from_validation']:.4f}")
print("test:", {k: results["test"][k] for k in ["pr_auc", "roc_auc", "precision", "recall", "f1"]})

## 6. Compare E0 vs E1 vs E2

The controlled experiment: the same model, seed and settings trained separately on each level, sharing the one
split/sample from step 3. Any difference in the table below comes from the data preparation.

In [ ]:
from src.evaluation.experiment import comparison_table, run_comparison, save_experiment

comparison = run_comparison(preparer, ["E0", "E1", "E2"], cfg, settings={}, device=hw["device"],
                            progress=lambda i, n, r: print(f"  {r.level} done ({i}/{n}): test PR-AUC {r.metrics['test']['pr_auc']}"))
comparison_table(comparison)

In [ ]:
# Optional: persist this comparison the same way the app's Experiments page does (experiments/experiment_vNNN.json),
# so it can be reopened later on the Results page or in another notebook run.
path = save_experiment(comparison, ROOT, dataset_id, split_info)
print(f"saved to {path}")

## Next steps

* Try a real dataset: set `PATH` in step 1 to your own CSV/JSON file.
* Try the optional Hugging Face backend (`src/models/hf_adapter.py`) — needs `pip install transformers peft
  accelerate` and, for anything beyond a tiny demo run, a GPU runtime.
* Open `PROJECT_STATUS.md` in the repository for exactly what's implemented, what was verified and how, and
  known limitations.